# 09 · Wiggle scoring (GPU) — dirty / U-Net / DDRM / beam-only, corrected methodology

Same comparison as `experiments/wiggle_all_methods.py`, on Kaggle GPU instead of local CPU.
Local CPU run of this exact script was still on its first channel-sampling config after two
hours; DDRM's diffusion sampling is the bottleneck and this notebook exists only to get a GPU
under it. Uses the fixed `compare_wiggles()` (one shared Keplerian model, fit on clean, reused
for every method — see RETRACTION_wiggle_methodology.md for why the old per-method-fit version
was wrong).

**Attach the Kaggle Dataset containing**: `clean_sg.fits`, `dirty_sg.fits`,
`dirty_beam_recovered_v2.fits`, `winner_aug_seed43.ckpt`, `ddrm_prior.ckpt`.

## 0. Bootstrap (clone repo for `src/`, locate the data files)

In [6]:
import os, sys, subprocess, glob

ON_KAGGLE = os.path.exists('/kaggle')
BRANCH = 'midterm-prep'
if ON_KAGGLE:
    REPO = '/kaggle/working/EXXA'; PKG = os.path.join(REPO, 'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',
                        'https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/'+BRANCH], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps',
                    'pytorch-msssim','bettermoments'], check=True)
    os.chdir(os.path.join(PKG,'notebooks')); sys.path.insert(0, PKG)

    hits = glob.glob('/kaggle/input/**/clean_sg.fits', recursive=True)
    if not hits:
        raise FileNotFoundError('No clean_sg.fits under /kaggle/input -- attach the wiggle-scoring Dataset.')
    DATA_DIR = os.path.dirname(hits[0])
else:
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'):
        os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
    DATA_DIR = '../self-gravitating cube and dirty cube/kinematic_data_v2'
print('DATA_DIR:', DATA_DIR)

From https://github.com/KrishanYadav333/EXXA
 * branch            midterm-prep -> FETCH_HEAD
   4ee37e9..9f59221  midterm-prep -> origin/midterm-prep


HEAD is now at 9f59221 results: wiggle_all_methods confirms the retracted correction, reproduced
DATA_DIR: /kaggle/input/datasets/krishanyadav333/kaggle-wiggle-scoring-dataset/kaggle-wiggle-scoring-dataset


## 0b. Pull latest code (re-run anytime, no kernel restart needed)

Cell 0b updates `src/` only, never this notebook's own cells (RULES.md #2). Kaggle's own
GitHub pull updates the cells; this cell just hot-reloads the library in between.

In [7]:
if ON_KAGGLE:
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', 'FETCH_HEAD'], check=True)
    print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'],
                         capture_output=True, text=True).stdout)
    import importlib, src
    importlib.reload(src)

HEAD is now at 9f59221 results: wiggle_all_methods confirms the retracted correction, reproduced
9f59221 results: wiggle_all_methods confirms the retracted correction, reproduced



From https://github.com/KrishanYadav333/EXXA
 * branch            midterm-prep -> FETCH_HEAD


## 1. Imports and paths

In [8]:
import time
import numpy as np
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as Fn
from astropy.io import fits

from src.models.unet import UNet
from src.training.diffusion import DenoisingDiffusion, data_transform, inverse_data_transform
from src.training.ddrm import beam_transfer_function, ddrm_steps
from src.evaluation.forward_operator import apply_beam
from src.evaluation.moment_maps import generate_moment_maps, signal_mask
from src.evaluation.gi_wiggle import quadratic_moment1, compare_wiggles

UNET_CKPT = os.path.join(DATA_DIR, 'winner_aug_seed43.ckpt')
PRIOR_CKPT = os.path.join(DATA_DIR, 'ddrm_prior.ckpt')
BEAM_PATH = os.path.join(DATA_DIR, 'dirty_beam_recovered_v2.fits')
SIZE, MSTAR_BOUND = 256, 50.0
dev = "cuda" if torch.cuda.is_available() else "cpu"
print('device:', dev)

with fits.open(os.path.join(DATA_DIR, 'clean_sg.fits'), memmap=True) as h:
    hdr, cdata = h[0].header, h[0].data[:]
with fits.open(os.path.join(DATA_DIR, 'dirty_sg.fits'), memmap=True) as h:
    ddata = h[0].data[:]
beam = fits.getdata(BEAM_PATH).astype(np.float64)
AU = abs(hdr["CDELT1"]) * 3600.0 * hdr.get("DIST_PC", 140.0)

device: cuda


## 2. Denoising functions (U-Net, DDRM)

In [9]:
def unet_denoise(dirty):
    import math
    ck = torch.load(UNET_CKPT, map_location=dev, weights_only=False)
    net = UNet(in_channels=ck.get("in_channels", 1), out_channels=1,
               base_channels=ck["base_channels"], channel_multipliers=ck["channel_multipliers"],
               time_emb_dim=128, num_res_blocks=2, groups=math.gcd(8, ck["base_channels"]),
               beam_dim=ck.get("beam_dim", 0)).to(dev)
    net.load_state_dict(ck["model_state_dict"]); net.eval()
    C, H, W = dirty.shape
    los = dirty.reshape(C, -1).min(axis=1); his = dirty.reshape(C, -1).max(axis=1)
    rng = his - los
    out = np.empty_like(dirty)
    with torch.no_grad():
        for s in range(0, C, 8):
            blk = dirty[s:s + 8]
            lo, hi = los[s:s + 8], his[s:s + 8]
            n = np.where((hi - lo)[:, None, None] > 0,
                         (blk - lo[:, None, None]) / np.where((hi - lo) > 0, hi - lo, 1)[:, None, None], 0)
            t = torch.from_numpy(n)[:, None].float().to(dev)
            t = Fn.interpolate(t, (SIZE, SIZE), mode="bilinear", align_corners=False)
            p = net(t, torch.zeros(t.size(0), dtype=torch.long, device=dev), None)
            b = Fn.interpolate(p, (H, W), mode="bilinear", align_corners=False)[:, 0].cpu().numpy()
            for k in range(b.shape[0]):
                out[s + k] = b[k] * rng[s + k] + los[s + k] if rng[s + k] > 0 else np.full((H, W), los[s + k])
    return out


def ddrm_restore(dirty):
    ck = torch.load(PRIOR_CKPT, map_location=dev, weights_only=False)
    r = DenoisingDiffusion(config=ck["config"], device=dev, checkpoint_path="/tmp/u.pth")
    r._core.load_state_dict(ck["state_dict"])
    if ck.get("ema_helper") and r.use_ema:
        r.ema_helper.load_state_dict(ck["ema_helper"]); r.ema_helper.ema(r._core)
    r.model.eval()
    tf = beam_transfer_function(beam, (SIZE, SIZE), device=dev); tf = tf / tf.max()
    out = []
    for i in range(dirty.shape[0]):
        d = dirty[i]; lo, hi = float(d.min()), float(d.max())
        d01 = (d - lo) / (hi - lo) if hi > lo else np.zeros_like(d)
        t = torch.from_numpy(d01)[None, None].float().to(dev)
        t = Fn.interpolate(t, (SIZE, SIZE), mode="bilinear", align_corners=False)
        sy = float(hdr.get("RMS", 0.0013)) / max(hi - lo, 1e-12) * 2.0
        with torch.no_grad():
            xs, _ = ddrm_steps(data_transform(t),
                               list(range(0, r.num_timesteps, r.num_timesteps // 50)),
                               r.model, r.betas, tf, sigma_y=sy,
                               prediction_type=ck["prediction_type"])
        o = inverse_data_transform(xs[-1].to(dev))
        o = Fn.interpolate(o, d.shape, mode="bilinear", align_corners=False)
        out.append(o[0, 0].cpu().numpy() * (hi - lo) + lo)
    return np.stack(out)

## 3. Score both channel-sampling configs

In [10]:
os.makedirs('../results/self-gravitating', exist_ok=True)
os.makedirs('../experiments', exist_ok=True)

for step in (1, 4):
    CH = list(range(240, 361, step))
    velax = (hdr["CRVAL3"] + (np.array(CH) + 1 - hdr["CRPIX3"]) * hdr["CDELT3"]) * 1000.0
    clean = np.stack([np.asarray(cdata[c], np.float32) for c in CH])
    dirty = np.stack([np.asarray(ddata[c], np.float32) for c in CH])

    print("\n" + "=" * 78)
    print(f"CONFIG: channels 240-360 step {step}  ({len(CH)} channels, "
          f"dv = {abs(hdr['CDELT3'])*step:.3f} km/s)")
    print("=" * 78)

    t0 = time.time()
    cubes = {"clean": clean, "dirty": dirty,
             "beam-only": apply_beam(clean.astype(np.float64), beam),
             "U-Net": unet_denoise(dirty),
             "DDRM": ddrm_restore(dirty)}
    print(f"  (methods computed in {(time.time()-t0)/60:.1f} min)")

    m0, _, _ = generate_moment_maps("", data_velax=(clean.astype(np.float64), velax))
    mask = signal_mask(m0, frac=0.02)

    rows = {}
    for tag, cube in cubes.items():
        v0, _ = quadratic_moment1(cube.astype(np.float64), velax)
        rows[tag] = dict(m1=v0 / 1000.0)

    cmp = compare_wiggles({t: rows[t]["m1"] for t in rows}, mask, AU, reference="clean")
    g = cmp["clean"]["geom"]
    flag = " DEGEN" if g["mstar_msun"] > 0.9 * MSTAR_BOUND else ""
    print(f"\n  shared model (fit on clean): mstar={g['mstar_msun']:.3f} incl={g['incl_deg']:.1f}"
          f" pa={g['pa_deg']:.1f} vsys={g['vsys']:.3f}{flag}")

    print(f"\n  {'method':12s} {'residRMS':>9s} {'raw r':>8s} {'resid r':>9s}")
    for tag in cubes:
        rms = cmp[tag]["rms_kms"]
        if tag == "clean":
            print(f"  {tag:12s} {rms:9.3f} {'--':>8s} {'--':>9s}")
            continue
        ok = np.isfinite(rows["clean"]["m1"][mask]) & np.isfinite(rows[tag]["m1"][mask])
        raw = float(np.corrcoef(rows["clean"]["m1"][mask][ok], rows[tag]["m1"][mask][ok])[0, 1])
        print(f"  {tag:12s} {rms:9.3f} {raw:8.4f} {cmp[tag]['corr']:9.4f}")

    if step == 1:
        np.savez("../experiments/wiggle_all_methods_step1.npz", mask=mask,
                 **{f"{t}_m1": rows[t]["m1"] for t in rows},
                 **{f"{t}_r": cmp[t]["residual"] for t in cmp})
        fig, ax = plt.subplots(2, 5, figsize=(23, 9))
        vm = np.nanpercentile(np.abs(rows["clean"]["m1"][mask]), 98)
        vr = np.nanpercentile(np.abs(cmp["clean"]["residual"][mask]), 98)
        for i, tag in enumerate(cubes):
            im = ax[0, i].imshow(np.where(mask, rows[tag]["m1"], np.nan), cmap="RdBu_r",
                                 vmin=-vm, vmax=vm, origin="lower")
            ax[0, i].set_title(f"{tag}: M1"); plt.colorbar(im, ax=ax[0, i], fraction=0.046)
            im2 = ax[1, i].imshow(np.where(mask, cmp[tag]["residual"], np.nan), cmap="RdBu_r",
                                  vmin=-vr, vmax=vr, origin="lower")
            ax[1, i].set_title(f"{tag}: residual (RMS {cmp[tag]['rms_kms']:.2f})")
            plt.colorbar(im2, ax=ax[1, i], fraction=0.046)
        plt.suptitle("GI wiggle, all methods at identical config (240-360, step 1)", fontsize=13)
        plt.tight_layout()
        plt.savefig("../results/self-gravitating/wiggle_all_methods.png", dpi=120)
        print("\n  saved -> results/self-gravitating/wiggle_all_methods.png")


CONFIG: channels 240-360 step 1  (121 channels, dv = 0.033 km/s)
  (methods computed in 7.4 min)

  shared model (fit on clean): mstar=0.537 incl=32.8 pa=0.0 vsys=0.083

  method        residRMS    raw r   resid r
  clean            0.182       --        --
  dirty            0.170   0.9928    0.8907
  beam-only        0.168   0.9947    0.9198
  U-Net            0.169   0.9874    0.8040
  DDRM             0.302   0.9532    0.5835

  saved -> results/self-gravitating/wiggle_all_methods.png

CONFIG: channels 240-360 step 4  (31 channels, dv = 0.133 km/s)
  (methods computed in 1.9 min)

  shared model (fit on clean): mstar=0.550 incl=32.0 pa=178.6 vsys=0.079

  method        residRMS    raw r   resid r
  clean            1.435       --        --
  dirty            1.425   0.9939    0.9984
  beam-only        1.426   0.9945    0.9986
  U-Net            1.396   0.9880    0.9969
  DDRM             1.503   0.9567    0.9872
